# Лабораторная 3. Элементарный перцептрон. Ансамбль элементарных перцептронов

**Задание 9.** Реализуйте программно алгоритм обучения элементарного перцептрона
для распознавания растровых изображений. Можете использовать следующие варианты
реализации:

а) 10 элементарных перцептронов, классифицирующих цифры как цифры двух классов:
эта цифра/другая цифра;

б) один перцептрон, вместо ступенчатой функции использующий в качестве функции
активации функцию мягкого максимума, обучаемый с помощью стохастического
градиентного спуска.

Может быть использована обучающая выборка для заданий 3 и 4 главы 3.6 "Задания"
раздела "Метод ближайшего соседа".

**Задание 10.** Сформируйте ансамбль из обученных элементарных перцептронов из
предыдущего задания. Сравните для одной и той же выборки результаты классификации
растровых изображений с результатами, полученными методом ближайшего соседа и с
помощью отдельного классификатора из предыдущего задания.

## 1. Теория

### Данные

MNIST, та же подвыборка, что в лабораторной 1: 2000 обучающих и 300 тестовых
изображений $28 \times 28$ в градациях серого, по 200 и 30 изображений каждой
цифры. Изображение разворачивается в вектор из 784 яркостей, делённых на 255, и
дополняется первой компонентой $x_0 = 1$, вес при которой играет роль смещения.

### Алгоритм обучения перцептрона

Перцептрон относит $x$ к классу $\mathcal{H}_1$, если $\langle w, x\rangle > 0$, и к
классу $\mathcal{H}_2$ иначе. Объекты обучающей выборки пронумерованы и
предъявляются по циклу, $x(n) = x(n \bmod N)$, начальные веса $w(0) = 0$. Если
$x(n)$ классифицирован верно, веса не меняются, иначе

$$w(n+1) = w(n) + \eta\,x(n) \ \text{ при } x(n) \in \mathcal{H}_1, \qquad
w(n+1) = w(n) - \eta\,x(n) \ \text{ при } x(n) \in \mathcal{H}_2 .$$

Алгоритм останавливается, когда за полный проход по выборке, то есть за $N$
итераций подряд, не было ни одной коррекции. Для линейно неотделимых классов
такого прохода не будет, поэтому число проходов ограничено сверху. При $w(0) = 0$
коэффициент $\eta$ только умножает все веса на одно и то же число и на ответы не
влияет, поэтому берётся $\eta = 1$.

### Вариант а: десять перцептронов

Для каждой цифры $c$ обучается свой перцептрон: $\mathcal{H}_1$ это изображения
цифры $c$, $\mathcal{H}_2$ это изображения остальных цифр.

### Вариант б: перцептрон с мягким максимумом

Веса это десять векторов $w_0, \ldots, w_9$, по одному на цифру. Вероятность цифры $c$

$$p_c(x) = \frac{e^{\langle w_c, x\rangle}}{\sum_{j=0}^{9} e^{\langle w_j, x\rangle}},$$

ответ $\arg\max_c p_c(x) = \arg\max_c \langle w_c, x\rangle$. Обучение минимизирует
перекрёстную энтропию $L = -\sum_k \log p_{y_k}(x_k)$, её градиент

$$\nabla_{w_c} L = \sum_k \bigl(p_c(x_k) - [\,y_k = c\,]\bigr)\,x_k .$$

Стохастический градиентный спуск: на каждой эпохе объекты перемешиваются и
делятся на батчи по $B$ объектов, после каждого батча
$w_c \leftarrow w_c - \eta\,g_c$, где $g_c$ это градиент по батчу, делённый на $B$.

### Ансамбль

Ансамбль составляют десять обученных перцептронов варианта а. Объект относится к
цифре, перцептрон которой даёт наибольшее значение $\langle w_c, x\rangle$:

$$\hat y(x) = \arg\max_c \langle w_c, x\rangle .$$

Так разрешаются случаи, когда ответ "эта цифра" дали несколько перцептронов или
ни одного.

### Метод ближайшего соседа

С лучшими параметрами из лабораторной 1: евклидово расстояние и $k = 1$, объект
относится к цифре ближайшего обучающего изображения.

### Что перебирается

| Гиперпараметр | Значения |
|---|---|
| $\eta$, вариант б | 0.01, 0.05, 0.1, 0.5 |
| число эпох, вариант б | 10, 30, 60 |
| размер батча $B$, вариант б | 32 |

$\eta$ и число эпох выбираются по accuracy на последних 400 изображениях
обучающей выборки при обучении на первых 1600, затем перцептрон обучается на всей
обучающей выборке. У варианта а гиперпараметров нет: $\eta = 1$, $w(0) = 0$,
обучение до остановки, но не более 1000 проходов.

### Меры качества

Для перцептрона варианта а на его задаче "эта цифра / другая цифра": доля верных
ответов accuracy и полнота, то есть доля изображений цифры $c$, на которых он
ответил "эта цифра". Для распознавания десяти цифр: accuracy
$= \frac1n \sum_k [\,\hat y_k = y_k\,]$ и macro $F_1$, среднее по цифрам значение
$F_1 = 2PR/(P + R)$, где $P$ и $R$ это точность и полнота для цифры.

## 2. Данные

In [1]:
import os
import struct
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import f1_score

RNG = np.random.default_rng(20260905)


def read_idx(path):
    with open(path, 'rb') as f:
        magic, = struct.unpack('>I', f.read(4))
        dims = struct.unpack('>' + 'I' * (magic & 0xff), f.read(4 * (magic & 0xff)))
        return np.frombuffer(f.read(), dtype=np.uint8).reshape(dims)


def mnist_dir():
    for c in [os.environ.get('MNIST_DIR'),
              '../../bachelor/Perceptron/data_generation/MNIST/raw',
              'data/MNIST/raw']:
        if c and Path(c, 'train-images-idx3-ubyte').exists():
            return Path(c)
    raise FileNotFoundError('не найдены файлы MNIST')


def stratified(X, y, per_class, rng):
    """Подвыборка с одинаковым числом изображений каждой цифры."""
    idx = np.concatenate([rng.choice(np.flatnonzero(y == c), per_class, replace=False)
                          for c in range(10)])
    rng.shuffle(idx)
    return X[idx], y[idx]


def features(X):
    """Вектор из 784 яркостей в [0, 1] с первой компонентой x_0 = 1."""
    flat = X.reshape(len(X), -1) / 255.0
    return np.hstack([np.ones((len(flat), 1)), flat])


D = mnist_dir()
X_tr_img, y_tr = stratified(read_idx(D / 'train-images-idx3-ubyte'),
                            read_idx(D / 'train-labels-idx1-ubyte'), 200, RNG)
X_te_img, y_te = stratified(read_idx(D / 't10k-images-idx3-ubyte'),
                            read_idx(D / 't10k-labels-idx1-ubyte'), 30, RNG)
X_tr, X_te = features(X_tr_img), features(X_te_img)
print(f'обучающая {X_tr.shape}, тестовая {X_te.shape}')

обучающая (2000, 785), тестовая (300, 785)


## 3. Задание 9, вариант а: десять элементарных перцептронов

In [2]:
def train_perceptron(X, t, eta=1.0, max_passes=1000):
    """Алгоритм обучения перцептрона; t[i] = 1 для класса H1 и -1 для H2.
    Возвращает веса, число проходов по выборке и признак остановки."""
    w = np.zeros(X.shape[1])
    for p in range(1, max_passes + 1):
        corrections = 0
        for i in range(len(X)):
            s = X[i] @ w  # скалярное произведение <w, x>
            if t[i] == 1 and s <= 0:
                w = w + eta * X[i]
                corrections += 1
            elif t[i] == -1 and s > 0:
                w = w - eta * X[i]
                corrections += 1
        if corrections == 0:
            return w, p, True
    return w, max_passes, False

In [3]:
W = np.zeros((10, X_tr.shape[1]))  # строка c это веса перцептрона для цифры c
rows = []
for c in range(10):
    t_tr = np.where(y_tr == c, 1, -1)
    W[c], passes, stopped = train_perceptron(X_tr, t_tr)
    answer = np.where(X_te @ W[c] > 0, 1, -1)  # ответы на всех тестовых изображениях
    t_te = np.where(y_te == c, 1, -1)
    rows.append(dict(цифра=c, проходов=passes, остановился='да' if stopped else 'нет',
                     accuracy=np.mean(answer == t_te),
                     полнота=np.mean(answer[y_te == c] == 1)))
binary = pd.DataFrame(rows)
print('тестовая выборка, задача "эта цифра / другая цифра":')
print(binary.round(4).to_string(index=False))

тестовая выборка, задача "эта цифра / другая цифра":
 цифра  проходов остановился  accuracy  полнота
     0        54          да    0.9833   0.9333
     1        31          да    0.9833   0.9667
     2        45          да    0.9833   0.8667
     3       146          да    0.9367   0.7333
     4        41          да    0.9600   0.9000
     5       199          да    0.9433   0.7333
     6        34          да    0.9767   0.9000
     7        71          да    0.9733   0.9000
     8       217          да    0.8933   0.6333
     9       233          да    0.9367   0.7333


## 4. Задание 9, вариант б: перцептрон с мягким максимумом

In [4]:
def softmax(Z):
    """Мягкий максимум по строкам матрицы Z."""
    Z = Z - Z.max(axis=1, keepdims=True)  # сдвиг на максимум строки спасает exp от переполнения
    E = np.exp(Z)
    return E / E.sum(axis=1, keepdims=True)


def train_softmax(X, y, eta, epochs, batch, rng):
    """Стохастический градиентный спуск по батчам; строка c матрицы W это w_c."""
    W = np.zeros((10, X.shape[1]))
    Y = np.eye(10)[y]  # строка k: нули и единица на месте y[k]
    idx = np.arange(len(X))
    for _ in range(epochs):
        rng.shuffle(idx)
        for s in range(0, len(X), batch):
            b = idx[s:s + batch]  # номера объектов очередного батча
            P = softmax(X[b] @ W.T)  # строка k: вероятности p_c для k-го объекта батча
            # (P - Y).T @ X это сумма (p_c - [y = c]) x по батчу для всех c сразу
            W -= eta * (P - Y[b]).T @ X[b] / len(b)
    return W

In [5]:
X_fit, y_fit = X_tr[:1600], y_tr[:1600]
X_val, y_val = X_tr[1600:], y_tr[1600:]

rows = []
for eta in [0.01, 0.05, 0.1, 0.5]:
    for epochs in [10, 30, 60]:
        W_s = train_softmax(X_fit, y_fit, eta, epochs, 32, np.random.default_rng(2))
        rows.append(dict(eta=eta, epochs=epochs,
                         accuracy=np.mean((X_val @ W_s.T).argmax(1) == y_val)))
grid = pd.DataFrame(rows)
print('accuracy на отложенных 400 изображениях обучающей выборки:')
print(grid.pivot_table(index='eta', columns='epochs', values='accuracy').round(4).to_string())

best = grid.loc[grid.accuracy.idxmax()]
W_soft = train_softmax(X_tr, y_tr, best.eta, int(best.epochs), 32, np.random.default_rng(2))
print(f'\nвыбрано: eta = {best.eta}, эпох {int(best.epochs)}')

accuracy на отложенных 400 изображениях обучающей выборки:
epochs      10      30      60
eta                           
0.01    0.8300  0.8475  0.8575
0.05    0.8525  0.8675  0.8700
0.10    0.8625  0.8725  0.8650
0.50    0.8650  0.8650  0.8650

выбрано: eta = 0.1, эпох 30


## 5. Задание 10: ансамбль и сравнение с методом ближайшего соседа

In [6]:
pred_ens = (X_te @ W.T).argmax(1)  # номер перцептрона с наибольшим <w_c, x>
pred_soft = (X_te @ W_soft.T).argmax(1)  # мягкий максимум не меняет порядок оценок

pred_knn = np.zeros(len(X_te), dtype=int)
for i in range(len(X_te)):
    # из каждой строки X_tr вычитается X_te[i]; столбец x_0 = 1 в расстоянии не участвует
    d = np.sqrt(((X_tr[:, 1:] - X_te[i, 1:]) ** 2).sum(axis=1))
    pred_knn[i] = y_tr[np.argmin(d)]

rows = []
for name, pred in [('ансамбль 10 перцептронов', pred_ens),
                   ('метод ближайшего соседа, k = 1', pred_knn),
                   ('перцептрон с мягким максимумом', pred_soft)]:
    rows.append(dict(модель=name, accuracy=np.mean(pred == y_te),
                     macro_f1=f1_score(y_te, pred, average='macro')))
summary = pd.DataFrame(rows)
summary.to_csv('results.csv', index=False)
print('тестовая выборка, 300 изображений:')
print(summary.round(4).to_string(index=False))

тестовая выборка, 300 изображений:
                        модель  accuracy  macro_f1
      ансамбль 10 перцептронов    0.8433    0.8423
метод ближайшего соседа, k = 1    0.9300    0.9300
перцептрон с мягким максимумом    0.8800    0.8797
